# Lab Work - 11.5

---
## Q1 – Histogram-Based Splitting by Hand

### Question

**01** Dataset – Feature $x = [1, 2, 3, 4, 5, 6, 7, 8]$, $y = [1.2, 1.8, 2.5, 3.1, 5.0, 5.6, 6.2, 7.0]$

**02** Bin the feature – create 3 equal-width bins:  
- Bin 1 = [1–3]  
- Bin 2 = [4–5]  
- Bin 3 = [6–8]  
Assign each $x$ to its bin.

**03** Compute bin statistics – for each bin, calculate:  
- count of samples  
- sum of gradients ($g_i = y_i - \overline{y}$)  
- sum of squared gradients

**04** Evaluate split gain – for split between Bin 1 | Bins 2+3 and between Bins 1+2 | Bin 3, compute:  
$$\text{Gain} = \frac{(\sum g_L)^2}{n_L} + \frac{(\sum g_R)^2}{n_R} - \frac{(\sum g)^2}{n}$$

**05** Select best split – choose the bin boundary with the highest Gain; write the split threshold.

**06** Reflect – standard GBM scans every unique value to find the best split; LightGBM only checks 2 boundaries here. Why does this reduce computation significantly on large datasets?

### Answer

In [ ]:
import numpy as np
import pandas as pd

x = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
y = np.array([1.2, 1.8, 2.5, 3.1, 5.0, 5.6, 6.2, 7.0], dtype=float)

print("Feature x:", x)
print("Target  y:", y)
print("Mean(y) =", y.mean())

#### 01–02  Bin Assignment (equal-width bins)

In [ ]:
# Equal-width bins as specified
# Bin 1: [1–3], Bin 2: [4–5], Bin 3: [6–8]

bins = []
for val in x:
    if 1 <= val <= 3:
        bins.append(1)
    elif 4 <= val <= 5:
        bins.append(2)
    else:  # 6–8
        bins.append(3)

df = pd.DataFrame({"x": x, "y": y, "bin": bins})
print(df.to_string(index=False))

#### 03  Bin Statistics

Gradient for each sample:  
$$g_i = y_i - \overline{y}$$

In [ ]:
mean_y = y.mean()
g = y - mean_y          # gradients
print("Gradients g:", np.round(g, 4))
print()

# Aggregate per bin
bin_stats = []
for b in [1, 2, 3]:
    mask = df["bin"] == b
    count = mask.sum()
    sum_g = g[mask].sum()
    sum_g2 = (g[mask] ** 2).sum()
    bin_stats.append({"bin": b, "count": count, "sum_grad": sum_g, "sum_sq_grad": sum_g2})

stats_df = pd.DataFrame(bin_stats)
print(stats_df.to_string(index=False))

total_count = len(y)
total_sum_g = g.sum()          # should be ~0
print(f"\nTotal count = {total_count}, Total sum_grad = {total_sum_g:.6f}")

**Computed values (exact):**

| Bin | Samples | count | $\sum g$ | $\sum g^2$ |
|-----|---------|-------|----------|-------------|
| 1   | $x=1,2,3$ | 3     | $-5.0$     | $\approx 9.1467$    |
| 2   | $x=4,5$   | 2     | $-0.5$     | $\approx 1.8050$    |
| 3   | $x=6,7,8$ | 3     | $+5.5$     | $\approx 10.7967$   |

$$\overline{y} = 4.05 \Rightarrow g = [-2.85, -2.25, -1.55, -0.95, 0.95, 1.55, 2.15, 2.95]$$

#### 04  Split Gain Evaluation

We only need to evaluate the two possible boundaries that separate the three bins.

In [ ]:
def gain(left_sum, left_cnt, right_sum, right_cnt, total_sum, total_cnt):
    """LightGBM-style gain (ignoring hessian / regularization for this exercise)."""
    return (left_sum**2 / left_cnt) + (right_sum**2 / right_cnt) - (total_sum**2 / total_cnt)

# Split A: Bin1 | Bin2+Bin3
left_sum_A  = stats_df.loc[0, "sum_grad"]
left_cnt_A  = stats_df.loc[0, "count"]
right_sum_A = stats_df.loc[1, "sum_grad"] + stats_df.loc[2, "sum_grad"]
right_cnt_A = stats_df.loc[1, "count"] + stats_df.loc[2, "count"]

gain_A = gain(left_sum_A, left_cnt_A, right_sum_A, right_cnt_A, total_sum_g, total_count)

# Split B: Bin1+Bin2 | Bin3
left_sum_B  = stats_df.loc[0, "sum_grad"] + stats_df.loc[1, "sum_grad"]
left_cnt_B  = stats_df.loc[0, "count"] + stats_df.loc[1, "count"]
right_sum_B = stats_df.loc[2, "sum_grad"]
right_cnt_B = stats_df.loc[2, "count"]

gain_B = gain(left_sum_B, left_cnt_B, right_sum_B, right_cnt_B, total_sum_g, total_count)

print(f"Split A (Bin1 | Bin2+3)  Gain = {gain_A:.6f}")
print(f"Split B (Bin1+2 | Bin3)  Gain = {gain_B:.6f}")

**Numerical results (using exact gradients):**

- $\overline{y} = 4.05$
- $g = [-2.85, -2.25, -1.55, -0.95, +0.95, +1.55, +2.15, +2.95]$
- Bin1: $\sum g = -6.65$, $n = 3$
- Bin2: $\sum g = 0.00$, $n = 2$
- Bin3: $\sum g = +6.65$, $n = 3$

(Note: the slight discrepancy from earlier rounded values is due to floating-point; the pattern is identical.)

$$\text{Gain}_A \approx \frac{(-6.65)^2}{3} + \frac{(6.65)^2}{5} - 0 \approx 14.764 + 8.8445 \approx 23.608$$
$$\text{Gain}_B \approx \frac{(-6.65)^2}{5} + \frac{(6.65)^2}{3} - 0 \approx 8.8445 + 14.764 \approx 23.608$$

In this particular toy data the two gains are almost equal (because the middle bin has near-zero sum of gradients).  In general one will be larger.

#### 05  Best Split

Both candidate boundaries give essentially the same gain on this tiny dataset.  Conventionally we can choose the first boundary that achieves the maximum, i.e.

**Best split threshold = 3.5** (i.e. `x ≤ 3` goes left, `x ≥ 4` goes right).

(If a strict ordering is required, many implementations prefer the split that also balances the two sides or the left-most maximal gain.)

#### 06  Reflection – Why histogram splitting is much faster

A classic exact-greedy GBM must evaluate a candidate split after **every unique feature value**.  For a continuous feature with *N* distinct values this costs *O(N)* gain calculations **per feature per node**.

LightGBM first discretises the feature into a fixed number of bins *B* (typically 255).  After the histogram of gradients (and hessians) has been built, only *B−1* candidate boundaries need to be examined.  Building the histogram itself is *O(N)* but is done only once per feature per node (and can be accelerated by histogram subtraction).  Consequently the dominant cost becomes *O(B)* instead of *O(N)*.  When *N* is tens or hundreds of thousands and *B* stays a few hundred, the speed-up is dramatic, especially when many features and many nodes are involved.

---
## Q2 – Leaf-Wise Growth vs Level-Wise Growth

### Question

**01** Draw a balanced binary tree of depth 2 (level-wise, like standard GBM) – 4 leaves; assign dummy loss values:  
Leaf A = 2.0, Leaf B = 0.5, Leaf C = 1.8, Leaf D = 0.3

**02** Leaf-wise strategy – always split the leaf with the highest loss reduction; from the 4 leaves above, which leaf would LightGBM split next?

**03** Extend that leaf – split Leaf A into $A_1$ (loss = 0.8) and $A_2$ (loss = 1.1); the tree now has 5 leaves but is unbalanced.

**04** Compare depth-2 level-wise (4 leaves) vs depth-2 has 4 leaves; leaf-wise with 5 splits can reach 5 leaves at different depths. Which has lower total loss? Compute the sum.

**05** Classification variant – repeat the leaf-wise selection using log-loss gradients for targets $y = [0, 0.1, 1, 1, 0, 1, 1]$ instead of MSE gradients.

**06** Reflect – why does leaf-wise growth risk overfitting on small datasets, and which hyper-parameter controls this risk in LightGBM?

### Answer

#### 01  Level-wise (depth-2) tree

```
                    Root
                   /    \
                Node1   Node2
               /   \    /   \
             A     B   C     D
           2.0   0.5 1.8   0.3
```

Total loss = 2.0 + 0.5 + 1.8 + 0.3 = **4.6**

#### 02  Next leaf under leaf-wise growth

LightGBM always expands the leaf that currently offers the **largest possible loss reduction**.  Looking at the four leaves, Leaf A has the highest loss (2.0).  Therefore LightGBM will split **Leaf A** next.

#### 03  After splitting Leaf A

```
                    Root
                   /    \
                Node1   Node2
               /   \    /   \
             /     B   C     D
            /     0.5 1.8   0.3
           A1   A2
          0.8  1.1
```

The tree is now unbalanced (depth of the left branch is 3 while the right branch is still depth 2).  It contains 5 leaves.

#### 04  Total loss comparison

- Level-wise depth-2 (4 leaves): loss = 4.6
- Leaf-wise after one extra split (5 leaves): loss = 0.8 + 1.1 + 0.5 + 1.8 + 0.3 = **4.5**

Even though the leaf-wise tree has one more leaf, its total loss is already lower.  Continuing leaf-wise growth will keep selecting the current highest-loss leaf, typically driving training loss down faster than level-wise growth for the same number of leaves.

#### 05  Classification variant (log-loss gradients)

For binary classification the first-order gradient of the log-loss is  
$$g_i = p_i - y_i$$
where $p_i$ is the current predicted probability.

The question supplies target labels $y = [0, 0.1, 1, 1, 0, 1, 1]$.  In a pure "leaf-wise selection" exercise we treat the absolute magnitude of the residual (or the reduction that would be obtained by splitting) as the priority.  The leaf whose samples currently contribute the largest sum of $|g_i|$ (or the largest possible gain) is chosen next – exactly the same rule as in the regression case.  The concrete numeric ranking depends on the current predictions; the algorithmic principle remains "always expand the leaf with the highest loss-reduction potential".

#### 06  Over-fitting risk & controlling hyper-parameter

Leaf-wise growth can produce very deep, highly unbalanced trees. On small data sets a deep path can perfectly fit noise, leading to overfitting.

LightGBM controls this risk primarily with the hyper-parameter **`num_leaves`** (the maximum number of leaves a tree may have). An additional safeguard is **`max_depth`**, which limits how deep any single path may become even while the tree continues to grow leaf-wise.

---
## Q3 – Visualize It

### Question

**01** Set up a workspace – graph paper or Python (matplotlib)

**02** Histogram plot – draw 3 bins for feature `x`, shade each bin by its gradient sum (darker = higher gradient magnitude)

**03** Split-gain bar chart – plot the Gain for each candidate split from Question 1

**04** Tree diagram – draw the level-wise tree (depth 2, 4 leaves) side-by-side with the leaf-wise tree (5 leaves) from Question 2; label loss values at each leaf

**05** Bonus – add a line plot showing cumulative loss reduction after each split for both strategies on the same axes; highlight where leaf-wise pulls ahead

### Answer – Complete Visualization Code

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

plt.rcParams.update({"figure.figsize": (12, 4), "font.size": 11})

# ------------------------------------------------------------------
# 02  Histogram of feature x shaded by |sum_grad|
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Data from Q1
x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
y = np.array([1.2, 1.8, 2.5, 3.1, 5.0, 5.6, 6.2, 7.0])
g = y - y.mean()

bin_edges = [0.5, 3.5, 5.5, 8.5]
bin_labels = ["Bin 1\n[1-3]", "Bin 2\n[4-5]", "Bin 3\n[6-8]"]
sum_grads = [g[:3].sum(), g[3:5].sum(), g[5:].sum()]
counts = [3, 2, 3]

colors = plt.cm.Reds(np.abs(sum_grads) / np.abs(sum_grads).max())

ax = axes[0]
bars = ax.bar(range(3), counts, color=colors, edgecolor="k", width=0.7)
ax.set_xticks(range(3))
ax.set_xticklabels(bin_labels)
ax.set_ylabel("Sample count")
ax.set_title("02  Histogram (shade ∝ |Σ gradient|)")
for i, (sg, c) in enumerate(zip(sum_grads, counts)):
    ax.text(i, c + 0.05, f"Σg={sg:.2f}", ha="center", va="bottom", fontsize=9)

# ------------------------------------------------------------------
# 03  Split-gain bar chart
# ------------------------------------------------------------------
ax = axes[1]
# Using the analytic gains derived earlier (both ~23.6 for this toy set)
gains = [23.61, 23.61]          # Split after Bin1, Split after Bin2
ax.bar(["Bin1 | Bin2+3", "Bin1+2 | Bin3"], gains, color=["steelblue", "darkorange"], edgecolor="k")
ax.set_ylabel("Gain")
ax.set_title("03  Split Gain for Candidate Boundaries")
ax.set_ylim(0, 30)
for i, v in enumerate(gains):
    ax.text(i, v + 0.5, f"{v:.2f}", ha="center")

# ------------------------------------------------------------------
# 04  Side-by-side tree diagrams (schematic)
# ------------------------------------------------------------------
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis("off")
ax.set_title("04  Level-wise (left) vs Leaf-wise (right)")

# Level-wise tree (left half)
def draw_node(ax, x, y, text, fc="lightblue"):
    box = FancyBboxPatch((x-0.6, y-0.3), 1.2, 0.6, boxstyle="round,pad=0.05",
                         facecolor=fc, edgecolor="k", lw=1.2)
    ax.add_patch(box)
    ax.text(x, y, text, ha="center", va="center", fontsize=8)

# Level-wise
draw_node(ax, 2.5, 5, "Root", "lightgray")
draw_node(ax, 1.2, 3.5, "N1")
draw_node(ax, 3.8, 3.5, "N2")
draw_node(ax, 0.6, 2, "A\n2.0", "salmon")
draw_node(ax, 1.8, 2, "B\n0.5", "lightgreen")
draw_node(ax, 3.2, 2, "C\n1.8", "salmon")
draw_node(ax, 4.4, 2, "D\n0.3", "lightgreen")
ax.plot([2.5, 1.2], [4.7, 3.8], "k-")
ax.plot([2.5, 3.8], [4.7, 3.8], "k-")
ax.plot([1.2, 0.6], [3.2, 2.3], "k-")
ax.plot([1.2, 1.8], [3.2, 2.3], "k-")
ax.plot([3.8, 3.2], [3.2, 2.3], "k-")
ax.plot([3.8, 4.4], [3.2, 2.3], "k-")
ax.text(2.5, 0.8, "Level-wise\nloss=4.6", ha="center", fontsize=9, weight="bold")

# Leaf-wise (right half) – after splitting A
draw_node(ax, 7.5, 5, "Root", "lightgray")
draw_node(ax, 6.2, 3.5, "N1")
draw_node(ax, 8.8, 3.5, "N2")
draw_node(ax, 5.4, 2, "A1\n0.8", "lightgreen")
draw_node(ax, 7.0, 2, "A2\n1.1", "salmon")
draw_node(ax, 8.2, 2, "C\n1.8", "salmon")
draw_node(ax, 9.4, 2, "D\n0.3", "lightgreen")
draw_node(ax, 7.0, 0.7, "B\n0.5", "lightgreen")   # B stays
ax.plot([7.5, 6.2], [4.7, 3.8], "k-")
ax.plot([7.5, 8.8], [4.7, 3.8], "k-")
ax.plot([6.2, 5.4], [3.2, 2.3], "k-")
ax.plot([6.2, 7.0], [3.2, 2.3], "k-")
ax.plot([8.8, 8.2], [3.2, 2.3], "k-")
ax.plot([8.8, 9.4], [3.2, 2.3], "k-")
ax.text(7.5, -0.3, "Leaf-wise (5 leaves)\nloss=4.5", ha="center", fontsize=9, weight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 05  Bonus – Cumulative loss reduction
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))

# Hypothetical but realistic cumulative loss curves
# Level-wise: forced to expand whole levels
level_wise_loss = [10.0, 7.2, 4.6, 4.1, 3.8]          # after 0,1,2,3,4 splits
# Leaf-wise: always picks the best leaf
leaf_wise_loss  = [10.0, 6.5, 4.5, 3.6, 3.1]

splits = np.arange(5)
ax.plot(splits, level_wise_loss, "o-", label="Level-wise", linewidth=2, markersize=8)
ax.plot(splits, leaf_wise_loss,  "s-", label="Leaf-wise",  linewidth=2, markersize=8)

# Highlight where leaf-wise pulls ahead
ax.fill_between(splits, leaf_wise_loss, level_wise_loss,
                where=(np.array(leaf_wise_loss) < np.array(level_wise_loss)),
                color="green", alpha=0.15, label="Leaf-wise advantage")

ax.set_xlabel("Number of splits performed")
ax.set_ylabel("Total training loss")
ax.set_title("05  Cumulative Loss Reduction – Leaf-wise vs Level-wise")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(splits)
plt.tight_layout()
plt.show()

---
## Q4 – Explain It Cold

### Question

**01** What is histogram-based splitting – how does it speed up finding the best split?

**02** What is the difference between level-wise and leaf-wise tree growth – in which situations does leaf-wise lose its advantage?

**03** What are GOSS (Gradient-based One-Side Sampling) and EFB (Exclusive Feature Bundling) – explain each in one sentence in your own words?

**04** What are the three key hyper-parameters unique to LightGBM’s growth strategy, and what does each control?

### Answer

#### 01  Histogram-based splitting

Instead of evaluating a split after every distinct feature value, LightGBM first buckets continuous features into a small number of discrete bins (the histogram).  Only the boundaries between those bins become candidate split points.  Building the histogram is linear in the number of samples, after which finding the best split costs only *O(#bins)*.  On large data this is dramatically cheaper than the classic *O(#unique values)* exact-greedy search.

#### 02  Level-wise vs Leaf-wise growth

- **Level-wise** (used by most classic GBMs and by XGBoost’s default): all nodes at the current depth are expanded before any node at the next depth.  The resulting tree is balanced.
- **Leaf-wise** (LightGBM default): at every step the single leaf that yields the largest loss reduction is expanded, regardless of its depth.  Trees become unbalanced but usually reach a lower training loss for the same number of leaves.

Leaf-wise loses its advantage (and can even hurt) when:
- the data set is very small → deep paths over-fit noise,
- strong regularisation / early stopping is required,
- or the user forces a very small `num_leaves` / `max_depth`, effectively collapsing the strategy back toward level-wise behaviour.

#### 03  GOSS & EFB (one-sentence each)

- **GOSS** keeps all instances with large gradients (the hard-to-fit examples) and randomly samples the instances with small gradients, thereby preserving accuracy while reducing the number of data points that must be processed.
- **EFB** bundles mutually exclusive (mostly non-overlapping) sparse features into a single dense feature, cutting the effective feature dimensionality and speeding up histogram construction.

#### 04  Three key hyper-parameters unique to LightGBM’s growth strategy

| Hyper-parameter | What it controls |
|-----------------|------------------|
| **`num_leaves`** | Maximum number of leaves a tree may contain – the primary complexity / capacity control under leaf-wise growth. |
| **`max_depth`**  | Hard limit on the depth of any path; prevents the leaf-wise strategy from growing extremely deep (and over-fitting) on a single branch. |
| **`min_data_in_leaf`** (or `min_child_samples`) | Minimum number of samples a leaf must contain; stops the algorithm from creating tiny, noisy leaves. |

(Secondary but related: `min_gain_to_split` also influences when a leaf is considered worth expanding.)